In [1]:
%pip install -q transformers accelerate rank-bm25 peft trl datasets pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 15.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os

DATA_ROOT = "/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data"

print("Dataset exists:", os.path.exists(DATA_ROOT))
print("\nFiles and folders:")

for root, dirs, files in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, "").count(os.sep)
    if level > 3:
        dirs[:] = []
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{indent}    {file} ({size_mb:.2f} MB)")


Dataset exists: True

Files and folders:
checkpoints-and-data/
    __huggingface_repos__.json (0.00 MB)
    search_rl_pretraining_backup/
        project_manifest.json (0.00 MB)
        nq_splits/
            test.jsonl (6.56 MB)
            train.jsonl (119.87 MB)
            validation.jsonl (6.72 MB)
        wiki/
            wikipedia_passages.jsonl (354.56 MB)
    search_rl_project/
        project_manifest.json (0.00 MB)
        nq_splits/
            test.jsonl (6.56 MB)
            train.jsonl (119.87 MB)
            validation.jsonl (6.72 MB)
        wiki/
            wikipedia_passages.jsonl (354.56 MB)


In [3]:

import os
import json
import shutil

DATA_ROOT = "/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data"

BACKUP_ROOT = os.path.join(DATA_ROOT, "search_rl_pretraining_backup")
PROJECT_DIR = "/kaggle/working/search_rl_project"

# Restore saved project files into writable storage
os.makedirs(PROJECT_DIR, exist_ok=True)

for folder in ["nq_splits", "wiki"]:
    src = os.path.join(BACKUP_ROOT, folder)
    dst = os.path.join(PROJECT_DIR, folder)

    if os.path.exists(dst):
        shutil.rmtree(dst)

    shutil.copytree(src, dst)

# Copy manifest
shutil.copy2(
    os.path.join(BACKUP_ROOT, "project_manifest.json"),
    os.path.join(PROJECT_DIR, "project_manifest.json")
)

# Define paths
NQ_DIR = os.path.join(PROJECT_DIR, "nq_splits")
WIKI_PASSAGES_PATH = os.path.join(PROJECT_DIR, "wiki", "wikipedia_passages.jsonl")

TRAIN_PATH = os.path.join(NQ_DIR, "train.jsonl")
VAL_PATH = os.path.join(NQ_DIR, "validation.jsonl")
TEST_PATH = os.path.join(NQ_DIR, "test.jsonl")

print("Project restored successfully!")
print("Project directory:", PROJECT_DIR)
print("Wikipedia passages:", os.path.exists(WIKI_PASSAGES_PATH))
print("Train file:", os.path.exists(TRAIN_PATH))
print("Validation file:", os.path.exists(VAL_PATH))
print("Test file:", os.path.exists(TEST_PATH))


Project restored successfully!
Project directory: /kaggle/working/search_rl_project
Wikipedia passages: True
Train file: True
Validation file: True
Test file: True


In [4]:

def count_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

for name, path in [
    ("Train", TRAIN_PATH),
    ("Validation", VAL_PATH),
    ("Test", TEST_PATH),
    ("Wikipedia passages", WIKI_PASSAGES_PATH)
]:
    print(f"{name}: {count_jsonl(path):,}")


Train: 96,223
Validation: 5,346
Test: 5,346
Wikipedia passages: 469,288


In [5]:

import re
import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

# Load Wikipedia passages
wiki_records = []

with open(WIKI_PASSAGES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        wiki_records.append(json.loads(line))

print("Loaded passages:", len(wiki_records))

# Combine title and passage text for indexing
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

wiki_documents = [
    (record.get("title", "") + " " + record.get("text", "")).strip()
    for record in wiki_records
]

tokenized_corpus = [
    tokenize(document)
    for document in tqdm(wiki_documents, desc="Tokenizing Wikipedia")
]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("\nBM25 index created!")
print("Indexed passages:", len(tokenized_corpus))


Loaded passages: 469288


Tokenizing Wikipedia:   0%|          | 0/469288 [00:00<?, ?it/s]


BM25 index created!
Indexed passages: 469288


In [6]:

def search_wikipedia(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "title": wiki_records[idx].get("title", ""),
            "text": wiki_records[idx].get("text", ""),
            "score": float(scores[idx])
        })

    return results


test_query = "Who played the mother in How I Met Your Mother?"

results = search_wikipedia(test_query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Title:", result["title"])
    print("Score:", round(result["score"], 4))
    print("Passage:", result["text"][:500])



Result 1
Title: Father's Day (disambiguation)
Score: 42.9251
Passage: tornado, also known as the Father's Day Tornado Les Compères, a 1983 French film remade as the 1997 film Father's Day Stepfather III, or Stepfather III: Father's Day in home video releases "Not a Father's Day", an episode of How I Met Your Mother Parents' Day, a holiday combining Father's Day and Mother's Day Mother's Day (disambiguation)

Result 2
Title: Like a Surgeon
Score: 39.944
Passage: show How I Met Your Mother. In one episode, lead character Ted Mosby gave "Weird Al" the idea to the song after sending him a fan letter when he was 8 years old. The creation of "Like a Surgeon", partially from Madonna's suggestion, became part of several scenes in the 2010 satirical Funny or Die web short, "Weird: The Al Yankovic Story" in which Yankovic (played by Aaron Paul) ends up in a love affair with Madonna (played by Olivia Wilde). The short, including this love affair, was later expand

Result 3
Title: Lindsey Alley
S

In [7]:

!pip install -q datasets tqdm

import os
import json
from tqdm.auto import tqdm
from datasets import load_dataset

# New corpus location; existing corpus remains untouched
WIKI_NEW_DIR = os.path.join(PROJECT_DIR, "wiki_100k")
os.makedirs(WIKI_NEW_DIR, exist_ok=True)

WIKI_NEW_PATH = os.path.join(
    WIKI_NEW_DIR,
    "wikipedia_100k_passages.jsonl"
)

# Load Wikipedia as a streaming dataset
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

# Shuffle the stream using a buffer
wiki_shuffled = wiki_stream.shuffle(
    seed=42,
    buffer_size=100_000
)

# Passage configuration
CHUNK_SIZE = 120
OVERLAP = 30
STRIDE = CHUNK_SIZE - OVERLAP
TARGET_ARTICLES = 100_000

article_count = 0
passage_count = 0

with open(WIKI_NEW_PATH, "w", encoding="utf-8") as output_file:

    for article in tqdm(
        wiki_shuffled.take(TARGET_ARTICLES),
        total=TARGET_ARTICLES,
        desc="Processing shuffled Wikipedia"
    ):

        title = article.get("title", "")
        text = article.get("text", "")
        article_id = article.get("id", "")

        if not text.strip():
            continue

        words = text.split()

        for start in range(0, len(words), STRIDE):

            chunk_words = words[start:start + CHUNK_SIZE]

            if not chunk_words:
                break

            passage = {
                "id": f"wiki100k-{article_id}-{start}",
                "title": title,
                "text": " ".join(chunk_words)
            }

            output_file.write(
                json.dumps(passage, ensure_ascii=False) + "\n"
            )

            passage_count += 1

            if start + CHUNK_SIZE >= len(words):
                break

        article_count += 1

print("\n===== NEW WIKIPEDIA CORPUS =====")
print("Articles processed:", article_count)
print("Passages created:", passage_count)
print(
    "File size (GB):",
    round(os.path.getsize(WIKI_NEW_PATH) / (1024**3), 2)
)
print("Saved at:", WIKI_NEW_PATH)


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Processing shuffled Wikipedia:   0%|          | 0/100000 [00:00<?, ?it/s]


===== NEW WIKIPEDIA CORPUS =====
Articles processed: 100000
Passages created: 429744
File size (GB): 0.31
Saved at: /kaggle/working/search_rl_project/wiki_100k/wikipedia_100k_passages.jsonl


In [8]:

WIKI_NEW_RECORDS = []

with open(WIKI_NEW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        WIKI_NEW_RECORDS.append(json.loads(line))

print("New passage count:", len(WIKI_NEW_RECORDS))

new_documents = [
    (r.get("title", "") + " " + r.get("text", "")).strip()
    for r in WIKI_NEW_RECORDS
]

new_tokenized_corpus = [
    tokenize(doc)
    for doc in tqdm(new_documents, desc="Tokenizing new corpus")
]

bm25_new = BM25Okapi(new_tokenized_corpus)

print("New BM25 index built successfully!")


New passage count: 429744


Tokenizing new corpus:   0%|          | 0/429744 [00:00<?, ?it/s]

New BM25 index built successfully!


In [9]:

import os
import json
import re
import random
import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

DATA_ROOT = "/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data"

PROJECT_DIR = "/kaggle/working/search_rl_project"

BACKUP_ROOT = os.path.join(
    DATA_ROOT, "search_rl_pretraining_backup"
)

NQ_DIR = os.path.join(PROJECT_DIR, "nq_splits")
WIKI_PASSAGES_PATH = os.path.join(
    PROJECT_DIR, "wiki", "wikipedia_passages.jsonl"
)

WIKI_NEW_PATH = os.path.join(
    PROJECT_DIR, "wiki_100k", "wikipedia_100k_passages.jsonl"
)

os.makedirs(PROJECT_DIR, exist_ok=True)

# Restore the original saved files if not already present
for folder in ["nq_splits", "wiki"]:
    src = os.path.join(BACKUP_ROOT, folder)
    dst = os.path.join(PROJECT_DIR, folder)

    if not os.path.exists(dst):
        import shutil
        shutil.copytree(src, dst)

print("Project paths restored.")


Project paths restored.


In [10]:

TRAIN_PATH = os.path.join(NQ_DIR, "train.jsonl")
VAL_PATH = os.path.join(NQ_DIR, "validation.jsonl")
TEST_PATH = os.path.join(NQ_DIR, "test.jsonl")

def read_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return records

validation_records = read_jsonl(VAL_PATH)

random.seed(42)

evaluation_records = random.sample(
    validation_records,
    min(100, len(validation_records))
)

print("Validation records:", len(validation_records))
print("Evaluation sample:", len(evaluation_records))


Validation records: 5346
Evaluation sample: 100


In [11]:

wiki_records = read_jsonl(WIKI_PASSAGES_PATH)
WIKI_NEW_RECORDS = read_jsonl(WIKI_NEW_PATH)

print("Original corpus:", len(wiki_records))
print("New corpus:", len(WIKI_NEW_RECORDS))


Original corpus: 469288
New corpus: 429744


In [12]:

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

# Original corpus
old_documents = [
    (r.get("title", "") + " " + r.get("text", "")).strip()
    for r in wiki_records
]

old_tokenized = [
    tokenize(doc)
    for doc in tqdm(old_documents, desc="Tokenizing original corpus")
]

bm25 = BM25Okapi(old_tokenized)

# New shuffled corpus
new_documents = [
    (r.get("title", "") + " " + r.get("text", "")).strip()
    for r in WIKI_NEW_RECORDS
]

new_tokenized_corpus = [
    tokenize(doc)
    for doc in tqdm(new_documents, desc="Tokenizing new corpus")
]

bm25_new = BM25Okapi(new_tokenized_corpus)

print("\nBoth BM25 indexes rebuilt successfully!")


Tokenizing original corpus:   0%|          | 0/469288 [00:00<?, ?it/s]

Tokenizing new corpus:   0%|          | 0/429744 [00:00<?, ?it/s]


Both BM25 indexes rebuilt successfully!


In [13]:

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def search_with_index(query, records, index, top_k=3):
    scores = index.get_scores(tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "title": records[idx]["title"],
            "text": records[idx]["text"],
            "score": float(scores[idx])
        }
        for idx in top_indices
    ]


def evaluate_retriever(records, index, eval_data, top_k=3):
    hits = 0
    reciprocal_ranks = []

    for record in tqdm(eval_data):
        answers = [
            normalize_text(a)
            for a in record.get("answers", [])
            if normalize_text(a)
        ]

        retrieved = search_with_index(
            record["question"], records, index, top_k
        )

        rank_found = None

        for rank, passage in enumerate(retrieved, start=1):
            text = normalize_text(
                passage["title"] + " " + passage["text"]
            )

            if any(answer in text for answer in answers):
                rank_found = rank
                break

        if rank_found is not None:
            hits += 1
            reciprocal_ranks.append(1 / rank_found)
        else:
            reciprocal_ranks.append(0)

    return (
        hits / len(eval_data),
        sum(reciprocal_ranks) / len(eval_data)
    )


In [14]:

old_recall, old_mrr = evaluate_retriever(
    wiki_records, bm25, evaluation_records
)

new_recall, new_mrr = evaluate_retriever(
    WIKI_NEW_RECORDS, bm25_new, evaluation_records
)

print("\n===== RETRIEVER COMPARISON =====")
print(f"Original 50K: Recall@3={old_recall:.4f}, MRR@3={old_mrr:.4f}")
print(f"Shuffled 100K: Recall@3={new_recall:.4f}, MRR@3={new_mrr:.4f}")


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]


===== RETRIEVER COMPARISON =====
Original 50K: Recall@3=0.1000, MRR@3=0.0783
Shuffled 100K: Recall@3=0.1000, MRR@3=0.0900


In [15]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.0 MB/s eta 0:00:00


In [16]:
import os
import json
import numpy as np
import torch
import faiss

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [17]:
print("New Wikipedia corpus:", len(WIKI_NEW_RECORDS))
print("Evaluation questions:", len(evaluation_records))

print("\nSample passage:")
print("Title:", WIKI_NEW_RECORDS[0]["title"])
print("Text:", WIKI_NEW_RECORDS[0]["text"][:300])

print("\nSample question:")
print(evaluation_records[0]["question"])

New Wikipedia corpus: 429744
Evaluation questions: 100

Sample passage:
Title: Nizhneye Kazanishche
Text: Nizhneye Kazanishche (; , Töben Qazanış) is a rural locality (a selo) in Buynaksky District, Republic of Dagestan, Russia. The population was 12,871 as of 2010. There are 162 streets. Geography Nizhneye Kazanishche is located 9 km southeast of Buynaksk (the district's administrative centre) by road,

Sample question:
who came up with the process of natural selection


In [18]:
E5_MODEL_NAME = "intfloat/e5-small-v2"

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device=DEVICE
)

print("E5 model loaded successfully!")
print("Embedding dimension:", e5_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

E5 model loaded successfully!
Embedding dimension: 384


/tmp/ipykernel_23/3181665308.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", e5_model.get_sentence_embedding_dimension())


In [19]:
RETRIEVAL_DIR = os.path.join(PROJECT_DIR, "retrieval_artifacts")
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

EMBEDDINGS_PATH = os.path.join(
    RETRIEVAL_DIR, "e5_100k_embeddings.npy"
)

# Prepare the passage texts
e5_documents = [
    "passage: " + (
        record.get("title", "") + ". " +
        record.get("text", "")
    ).strip()
    for record in WIKI_NEW_RECORDS
]

print("Total passages:", len(e5_documents))
print("Example:", e5_documents[0][:250])

Total passages: 429744
Example: passage: Nizhneye Kazanishche. Nizhneye Kazanishche (; , Töben Qazanış) is a rural locality (a selo) in Buynaksky District, Republic of Dagestan, Russia. The population was 12,871 as of 2010. There are 162 streets. Geography Nizhneye Kazanishche is l


In [20]:
NUM_PASSAGES = len(e5_documents)
EMBEDDING_DIM = e5_model.get_sentence_embedding_dimension()

# Create a memory-mapped file to avoid holding
# the complete embedding matrix in RAM.
passage_embeddings = np.lib.format.open_memmap(
    EMBEDDINGS_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(NUM_PASSAGES, EMBEDDING_DIM)
)

BATCH_SIZE = 64

for start in tqdm(
    range(0, NUM_PASSAGES, BATCH_SIZE),
    desc="Generating E5 embeddings"
):
    end = min(start + BATCH_SIZE, NUM_PASSAGES)

    batch_texts = e5_documents[start:end]

    batch_embeddings = e5_model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    passage_embeddings[start:end] = batch_embeddings.astype(
        np.float32
    )

passage_embeddings.flush()

print("\nEmbedding generation completed!")
print("Embedding shape:", passage_embeddings.shape)
print(
    "Embedding file size:",
    round(os.path.getsize(EMBEDDINGS_PATH) / (1024**3), 2),
    "GB"
)

/tmp/ipykernel_23/1060777787.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = e5_model.get_sentence_embedding_dimension()


Generating E5 embeddings:   0%|          | 0/6715 [00:00<?, ?it/s]


Embedding generation completed!
Embedding shape: (429744, 384)
Embedding file size: 0.61 GB


In [21]:
e5_index = faiss.IndexFlatIP(EMBEDDING_DIM)

e5_index.add(np.asarray(passage_embeddings))

print("FAISS index created!")
print("Total indexed passages:", e5_index.ntotal)

FAISS index created!
Total indexed passages: 429744


In [22]:
FAISS_INDEX_PATH = os.path.join(
    RETRIEVAL_DIR, "e5_100k_index.faiss"
)

faiss.write_index(e5_index, FAISS_INDEX_PATH)

print("FAISS index saved:", FAISS_INDEX_PATH)

FAISS index saved: /kaggle/working/search_rl_project/retrieval_artifacts/e5_100k_index.faiss


In [23]:
def search_e5(query, top_k=3):

    query_text = "query: " + query

    query_embedding = e5_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = e5_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "id": WIKI_NEW_RECORDS[idx]["id"],
            "title": WIKI_NEW_RECORDS[idx]["title"],
            "text": WIKI_NEW_RECORDS[idx]["text"],
            "score": float(score)
        })

    return results

In [24]:
test_question = "Who came up with the process of natural selection?"

results = search_e5(test_question, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Title:", result["title"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:400])


Result 1
Title: Anthropogeny
Score: 0.8468
Text: Anthropogeny is the study of human origins. It is not simply a synonym for human evolution by natural selection, which is only a part of the processes involved in human origins. Many other factors besides natural selection were involved, ranging over climatic, geographic, ecological, social, and cultural ones. Anthropogenesis, meaning the process or point of becoming human, is also called hominiza

Result 2
Title: Lymphater's Formula
Score: 0.8407
Text: ability to apriori knowledge, i.e., knowledge neither inherited nor learned. He decided to consult a famous myrmecologist, who told him about a rare ant species Acanthis Rubra Willinsoniana with an exceptionally high adaptability... Eventually Lymphater devised and constructed "It" capable of instant precognition of everything within "Its" rapidly expanding range of perception. From "It" Lymphater

Result 3
Title: Darwin's Darkest Hour
Score: 0.8354
Text: theory of evolution existing thr

In [25]:
import inspect

print(inspect.signature(evaluate_retriever))
print(inspect.getsource(evaluate_retriever))

(records, index, eval_data, top_k=3)
def evaluate_retriever(records, index, eval_data, top_k=3):
    hits = 0
    reciprocal_ranks = []

    for record in tqdm(eval_data):
        answers = [
            normalize_text(a)
            for a in record.get("answers", [])
            if normalize_text(a)
        ]

        retrieved = search_with_index(
            record["question"], records, index, top_k
        )

        rank_found = None

        for rank, passage in enumerate(retrieved, start=1):
            text = normalize_text(
                passage["title"] + " " + passage["text"]
            )

            if any(answer in text for answer in answers):
                rank_found = rank
                break

        if rank_found is not None:
            hits += 1
            reciprocal_ranks.append(1 / rank_found)
        else:
            reciprocal_ranks.append(0)

    return (
        hits / len(eval_data),
        sum(reciprocal_ranks) / len(eval_data)
    )



In [26]:
def evaluate_retriever(search_function, eval_data, top_k=3):

    hits = 0
    reciprocal_ranks = []

    for record in tqdm(eval_data):

        answers = [
            normalize_text(a)
            for a in record.get("answers", [])
            if normalize_text(a)
        ]

        retrieved = search_function(
            record["question"], top_k=top_k
        )

        rank_found = None

        for rank, passage in enumerate(retrieved, start=1):

            text = normalize_text(
                passage["title"] + " " + passage["text"]
            )

            if any(answer in text for answer in answers):
                rank_found = rank
                break

        if rank_found is not None:
            hits += 1
            reciprocal_ranks.append(1 / rank_found)
        else:
            reciprocal_ranks.append(0)

    return (
        hits / len(eval_data),
        sum(reciprocal_ranks) / len(eval_data)
    )

In [27]:
e5_recall, e5_mrr = evaluate_retriever(
    search_e5,
    evaluation_records,
    top_k=3
)

  0%|          | 0/100 [00:00<?, ?it/s]

In [28]:
def search_bm25_new(query, top_k=3):
    return search_with_index(
        query,
        WIKI_NEW_RECORDS,
        bm25_new,
        top_k
    )

bm25_recall, bm25_mrr = evaluate_retriever(
    search_bm25_new,
    evaluation_records,
    top_k=3
)

  0%|          | 0/100 [00:00<?, ?it/s]

In [29]:
print("\n===== RETRIEVER COMPARISON =====")

print(
    f"BM25 100K: Recall@3={bm25_recall:.4f}, "
    f"MRR@3={bm25_mrr:.4f}"
)

print(
    f"E5 100K:   Recall@3={e5_recall:.4f}, "
    f"MRR@3={e5_mrr:.4f}"
)


===== RETRIEVER COMPARISON =====
BM25 100K: Recall@3=0.1000, MRR@3=0.0900
E5 100K:   Recall@3=0.1200, MRR@3=0.0700


In [30]:
def check_retrieval(record, search_function, top_k=3):

    answers = [
        normalize_text(a)
        for a in record.get("answers", [])
        if normalize_text(a)
    ]

    retrieved = search_function(
        record["question"], top_k=top_k
    )

    for rank, passage in enumerate(retrieved, start=1):

        text = normalize_text(
            passage["title"] + " " + passage["text"]
        )

        if any(answer in text for answer in answers):
            return rank

    return None


comparison = []

for record in tqdm(evaluation_records):

    bm25_rank = check_retrieval(
        record, search_bm25_new
    )

    e5_rank = check_retrieval(
        record, search_e5
    )

    comparison.append({
        "question": record["question"],
        "answers": record.get("answers", []),
        "bm25_rank": bm25_rank,
        "e5_rank": e5_rank
    })


e5_only = [
    x for x in comparison
    if x["e5_rank"] is not None
    and x["bm25_rank"] is None
]

bm25_only = [
    x for x in comparison
    if x["bm25_rank"] is not None
    and x["e5_rank"] is None
]

both_success = [
    x for x in comparison
    if x["e5_rank"] is not None
    and x["bm25_rank"] is not None
]

print("E5-only successes:", len(e5_only))
print("BM25-only successes:", len(bm25_only))
print("Both successful:", len(both_success))

  0%|          | 0/100 [00:00<?, ?it/s]

E5-only successes: 6
BM25-only successes: 4
Both successful: 6


In [31]:
print("\n===== E5 SUCCEEDS, BM25 FAILS =====")

for item in e5_only[:5]:
    print("\nQuestion:", item["question"])
    print("Expected:", item["answers"])
    print("E5 rank:", item["e5_rank"])


print("\n===== BM25 SUCCEEDS, E5 FAILS =====")

for item in bm25_only[:5]:
    print("\nQuestion:", item["question"])
    print("Expected:", item["answers"])
    print("BM25 rank:", item["bm25_rank"])


===== E5 SUCCEEDS, BM25 FAILS =====

Question: who came up with the process of natural selection
Expected: ['Charles Darwin']
E5 rank: 2

Question: who sings this guys in love with you
Expected: ['Herb Alpert']
E5 rank: 1

Question: who decide a bill is a money bill
Expected: ['House of Representatives']
E5 rank: 3

Question: how many episodes are in crisis on earth x
Expected: ['four']
E5 rank: 2

Question: when is the penitential rite said in mass
Expected: ['the start']
E5 rank: 3

===== BM25 SUCCEEDS, E5 FAILS =====

Question: where is electron transport chain located in bacteria
Expected: ['membrane']
BM25 rank: 1

Question: when was the last time.the browns were in the playoffs
Expected: ['2002']
BM25 rank: 1

Question: what was the first year for the morgan silver dollar
Expected: ['1878']
BM25 rank: 2

Question: which nba team did lebron james go to
Expected: ['Lakers']
BM25 rank: 2


In [32]:
import os
import json

PROJECT_DIR = "/kaggle/working/search_rl_project"

artifacts = {
    "train_questions": TRAIN_PATH,
    "validation_questions": VAL_PATH,
    "test_questions": TEST_PATH,
    "wikipedia_100k": WIKI_NEW_PATH,
    "e5_embeddings": EMBEDDINGS_PATH,
    "faiss_index": FAISS_INDEX_PATH
}

manifest = {
    "project": "Search-R1 Resource-Constrained Implementation",
    "retriever": "intfloat/e5-small-v2",
    "embedding_dimension": 384,
    "similarity": "Cosine similarity using normalized vectors and FAISS IndexFlatIP",
    "wikipedia_passages": len(WIKI_NEW_RECORDS),
    "evaluation_questions": len(evaluation_records),
    "artifacts": {}
}

for name, path in artifacts.items():
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)

        manifest["artifacts"][name] = {
            "path": path,
            "size_gb": round(size_gb, 3)
        }

        print(f"{name}: {size_gb:.3f} GB")
    else:
        print(f"MISSING: {name} -> {path}")

manifest_path = os.path.join(PROJECT_DIR, "experiment_manifest.json")

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("\nManifest saved:", manifest_path)

train_questions: 0.117 GB
validation_questions: 0.007 GB
test_questions: 0.006 GB
wikipedia_100k: 0.305 GB
e5_embeddings: 0.615 GB
faiss_index: 0.615 GB

Manifest saved: /kaggle/working/search_rl_project/experiment_manifest.json
